# אימון גלאי לילה (night.pt) — Google Colab

לפני שמריצים: **Runtime → Change runtime type → T4 GPU**.

מה המחברת עושה: מורידה את ExDark (7,363 תמונות לילה מתויגות) מ-Hugging Face,
מוסיפה (אם יש) פריימים שתייגתן בעצמכן, מאמנת YOLOv8s ומורידה `best.pt`.

**מומלץ**: טוקן חינמי של Hugging Face, אחרת ההורדה נחסמת זמנית (429) ומתקדמת לאט.
huggingface.co → Settings → Access Tokens → New token (read). בקולאב: אייקון המפתח בצד
(Secrets) → הוסיפו `HF_TOKEN` והפעילו "Notebook access".

In [ ]:
# 1. setup
!pip -q install ultralytics huggingface_hub
import os, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none — switch runtime to GPU')
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN'); print('HF token: ok')
except Exception:
    print('no HF_TOKEN secret — download will be slower (rate limits)')

In [ ]:
# 2. the dataset script from the repo (or upload backend/train/night_dataset.py if this fails)
import urllib.request
URL = 'https://raw.githubusercontent.com/Talyaavraham/final_project/main/backend/train/night_dataset.py'
try:
    urllib.request.urlretrieve(URL, '/content/night_dataset.py'); print('got night_dataset.py from GitHub')
except Exception as e:
    print('download failed:', e); print('upload night_dataset.py:')
    from google.colab import files
    files.upload()
import sys; sys.path.insert(0, '/content'); import night_dataset; print(night_dataset.CLASSES)

In [ ]:
# 3. (optional) your own labelled frames: a Roboflow "YOLOv8" export zipped (data.yaml + train/images + train/labels)
#    Press Cancel to skip.
from google.colab import files
import os, shutil, zipfile
own = []
up = files.upload()
for name in up:
    shutil.rmtree('/content/own', ignore_errors=True)
    zipfile.ZipFile(name).extractall('/content/own')
    root = '/content/own'
    if not os.path.exists(root + '/data.yaml'):            # zip with one top folder
        subs = [d for d in os.listdir(root) if os.path.isdir(f'{root}/{d}')]
        root = f'{root}/{subs[0]}' if subs else root
    own = [root]; print('own export:', root, os.listdir(root))
print('own exports:', own or 'none')

In [ ]:
# 4. build the dataset: ExDark (~280 MB) + own frames (x4), classes in COCO names
import night_dataset as nd
sources = [('exdark', nd.download_exdark('/content/night/exdark'))]
repeat = {}
for i, d in enumerate(own):
    sources.append((f'own{i}', d)); repeat[f'own{i}'] = 4
nd.build('/content/night/dataset', sources, keep=nd.CLASSES, repeat=repeat)
print(open('/content/night/dataset/data.yaml').read())

In [ ]:
# 5. train. yolov8s, imgsz 800: ~2.5 min/epoch on a T4 -> 25 epochs ≈ 1 h.
#    In a hurry: epochs=12, or model='yolov8n.pt' (3x faster, a bit less accurate).
from ultralytics import YOLO
model = YOLO('yolov8s.pt')
model.train(data='/content/night/dataset/data.yaml', epochs=25, imgsz=800, batch=16, device=0,
            project='/content/runs', name='night', exist_ok=True, seed=0,
            close_mosaic=5, scale=0.6, hsv_v=0.5,      # scale/brightness jitter: small, dark people
            patience=8)

In [ ]:
# 6. metrics per class on the validation split
m = YOLO('/content/runs/night/weights/best.pt')
r = m.val(data='/content/night/dataset/data.yaml', imgsz=800, plots=False)
for i, n in r.names.items():
    print(f'{n:13s} mAP50={r.box.ap50[i]:.3f}  mAP50-95={r.box.maps[i]:.3f}')
print('all           mAP50=%.3f' % r.box.map50)

In [ ]:
# 7. quick look: predictions on your frames (if uploaded) or validation images
import glob, cv2
from IPython.display import Image, display
imgs = sorted(glob.glob('/content/own/**/*.jpg', recursive=True)) or sorted(glob.glob('/content/night/dataset/images/val/*.jpg'))
for p in imgs[:4]:
    res = m.predict(p, imgsz=800, conf=0.25, verbose=False)[0]
    cv2.imwrite('/content/pred.jpg', cv2.resize(res.plot(), (800, 450)))
    display(Image('/content/pred.jpg'))

In [ ]:
# 8. download best.pt  ->  save it as backend/night.pt
files.download('/content/runs/night/weights/best.pt')

## חזרה למחשב

```powershell
copy best.pt backend\night.pt
cd backend
$env:YOLO_MODEL = "yolov8n.pt,game.pt,night.pt"
$env:IMGSZ = "800"
.\venv\Scripts\python -m uvicorn main:app --port 8000
```

סרטון חשוך יקבל את `night.pt` אוטומטית (בחירת המודל לפי 20 פריימים).